# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'acm18'   # <-- set this
DATA = f'mdata/acm18/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
df = load('p1a_extinction')
am = df['airmass'].to_numpy()
zp = df['zeropoint_mag'].to_numpy()
sigmag = df['sigma_quoted_mag'].to_numpy()
def ols(A, y):
    AtA_inv = np.linalg.inv(A.T @ A)
    theta = AtA_inv @ A.T @ y
    resid = y - A @ theta
    s2 = resid @ resid / (len(y) - A.shape[1])
    return theta, s2 * AtA_inv, resid

ones = np.ones_like(am)
A = np.column_stack((ones, am))

theta, cov, res = ols(A, zp)
beta_hat, alpha_hat = theta
sig_beta, sig_alpha = np.sqrt(np.diag(cov))

print(f"OLS Fitted Parameters:\nk = {alpha_hat:.2f} ± {sig_alpha:.3f}\nm0 = {beta_hat:.2f} ± {sig_beta:.3f}")

In [ ]:
results,cov=np.polyfit(am, zp, 1, w=1/sigmag, cov=True)
k,m0 =results[0],results[1]
m0 = results[1]
sig_k, sig_m0 = np.sqrt(np.diag(cov))

print(f"np.polyfit Fitted Parameters:\nk = {k:.2f} ± {sig_k:.3f}\nm0 = {m0:.2f} ± {sig_m0:.3f}")

xx = np.linspace(1,2.5,500)

plt.errorbar(am,zp, yerr=sigmag, fmt='o', capsize=3, label='data')
plt.plot(xx, beta_hat + alpha_hat*(xx), label = 'OLS Fit')
plt.plot(xx,m0+k*xx,label='Weighted fit')

plt.title('ZeropointMag vs Airmass');plt.xlabel('Airmass');plt.ylabel('ZPt (mag)');plt.legend()

In this instance of np.polyfit the returned covarience matrix has been scalled by chi2/dof, where dof = M - (deg + 1). This treats the given weights as unreliable and the reported errors have been scalled to give the answer unity.

In [ ]:
'''Chi-square'''
# --------------------------------------------------
# Goodness of fit using quoted uncertainties
# --------------------------------------------------
model = m0+k*am
resid = zp-model

chi2 = np.sum((resid/sigmag)**2)
dof = len(zp)-2
chi2_red = chi2/dof
pvalue = stats.chi2.sf(chi2,dof)

print('chi2 = {:.2f}'.format(chi2))
print('dof = {}'.format(dof))
print('reduced chi2 = {:.2f}'.format(chi2_red))
print('p-value = {:.3g}'.format(pvalue))

The wighted linear fit gives $\chi^2 = 2.01$ with 13 dof correspunding to a reduced $\chi^2$ of 0.15. This tells me that the residual scatter is unusually small if the pipeline uncertainties are accurate. This then leads me to believe that the pipline uncertainties are overestimated leading to a deflated $\chi^2$ value. I belive that a value this small is extreemly unlikely if the quoted errors were correct.

**ANSWER:**

I report that $k = 0.24 \pm 0.006$. I choose this value because the quoted pipeline uncertainties appear overestimated, therefore the OLS based uncertainty should be more reprisentative of the actual scatter than the pipeline-weighted uncertainty.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
'''1b: Cramer-Rao bound'''

df1 = load('p1b_centroid')

psf_sigma = df1['psf_sigma_arcsec'].to_numpy()[0]
n_photons = df1['n_photons'].to_numpy()[0]
sig_quoted = df1['quoted_sigma_x0_arcsec'].to_numpy()[0]

fisher_per_photon = 1/psf_sigma**2
fisher_total = n_photons*fisher_per_photon
sig_cr = 1/np.sqrt(fisher_total)
ratio = sig_quoted/sig_cr

print('PSF sigma = {:.5f} arcsec'.format(psf_sigma))
print('N photons = {}'.format(n_photons))
print('Cramer-Rao bound = {:.5f} arcsec'.format(sig_cr))
print('Quoted uncertainty = {:.5f} arcsec'.format(sig_quoted))
print('Quoted / CR bound = {:.3f}'.format(ratio))

**ANSWER:**

The cramer-rao bound gives the lower limit on what the error can be. The quoted uncertainty if significantly smaller than the CR bound meaning that this error is not possible. 
The classmates quoted uncertainty cannot be valid under the gaussian independednt-photon model because it lies bellow the Cramer-Rao bound. Ways that they could have gone wrong is inputting a much larger value for the number of counts or theres a mathematical error in the calculation of the fisher matrix.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
'''1c.1: Curvature uncertainty'''
# --------------------------------------------------
# Quadratic fit to the -2 ln L profile
# --------------------------------------------------
df2 = load('p1c_profile')
depth = df2['depth_ppt'].to_numpy()
minus2lnL = df2['minus2lnL'].to_numpy()

a,b,c = np.polyfit(depth,minus2lnL,2)

depth_hat = -b/(2*a)
minus2lnL_min = np.polyval([a,b,c],depth_hat)

hessian_minus2lnL = 2*a
sig_correct = np.sqrt(2/hessian_minus2lnL)
sig_classmate = np.sqrt(1/hessian_minus2lnL)

print('Best-fit depth = {:.3f} ppt'.format(depth_hat))
print('Correct error from -2 ln L curvature = {:.3f} ppt'.format(sig_correct))
print("Classmate's error = {:.3f} ppt".format(sig_classmate))
print("Classmate's error is too small by a factor of {:.3f}.".format(sig_correct/sig_classmate))

My classmate's answer is off by a factor of 1.414 because it forgets to have sqrt(2) in the errors numerator. This is probably from treating the hessian an -ln(L) instead of the correct -2ln(L).

In [ ]:
'''1c.2: Profile-likelihood interval'''
# --------------------------------------------------
# Delta(-2 ln L) = 1 crossings
# --------------------------------------------------
i = np.argmin(minus2lnL)
depth_hat = depth[i]
target = minus2lnL[i]+1

lower = np.interp(target,minus2lnL[:i+1][::-1],depth[:i+1][::-1])
upper = np.interp(target,minus2lnL[i:],depth[i:])

sig_minus = depth_hat-lower
sig_plus = upper-depth_hat

plt.plot(depth,minus2lnL,'o-',ms=3)
plt.axhline(target,color='C1',ls='--',label=r'$\Delta(-2\ln L)=1$')
plt.axvline(depth_hat,color='k',ls='--',label='Best fit')
plt.axvline(lower,color='C2',ls=':')
plt.axvline(upper,color='C2',ls=':')
plt.xlabel('Depth (ppt)')
plt.ylabel(r'$-2\ln L$')
plt.legend()
plt.show()

print('Best-fit depth = {:.3f} ppt'.format(depth_hat))
print('68% profile interval = [{:.3f}, {:.3f}] ppt'.format(lower,upper))
print('Profile result = {:.3f} +{:.3f} -{:.3f} ppt'.format(depth_hat,sig_plus,sig_minus))

The $\Delta(-2\ln L) = 1$ interval appears to be symetric and only has a slight deviation in the thousands position. The seperation of these values from the computed curvature error is near exact at .243.

I expect my that simulated data using my classmates $\sigma$ will only contain the true depth 52% of the time. This is 100- 68/sqrt(2) because 1 $\sigma$ should include 68% of the data by the errors are off by a factor of sqrt(2)

In [ ]:
'''1c.3: Coverage simulation'''
# --------------------------------------------------
# Check the classmate's incorrectly quoted uncertainty
# --------------------------------------------------
rng = np.random.default_rng(457)
nsim = 100000

i = np.argmin(minus2lnL)
h = depth[i+1]-depth[i]
H = (minus2lnL[i+1]-2*minus2lnL[i]+minus2lnL[i-1])/h**2

sig_correct = np.sqrt(2/H)
sig_classmate = 1/np.sqrt(H)
depth_true = depth[i]

depth_hat_sim = rng.normal(depth_true,sig_correct,nsim)
covered = np.abs(depth_hat_sim-depth_true)<=sig_classmate

coverage = np.mean(covered)
mc_error = np.sqrt(coverage*(1-coverage)/nsim)
expected = 2*stats.norm.cdf(sig_classmate/sig_correct)-1

print("Correct sigma = {:.3f} ppt".format(sig_correct))
print("Classmate's quoted sigma = {:.3f} ppt".format(sig_classmate))
print("Predicted coverage = {:.1%}".format(expected))
print("Simulated coverage = {:.1%} +/- {:.1%}".format(coverage,mc_error))

**ANSWER:**

My prediction was spot on, the underestimated error bounds only contained true values 52% of the time.

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
df3 = load('p1d_mass_samples')
# median = df3['M2_Msun'].median()
# top = df3['M2_Msun'].quantile(.84)
# bottom = df3['M2_Msun'].quantile(.16)
q = df3['M2_Msun'].quantile([0.16,0.50,0.84]).to_numpy()
print('M2 = {:.2f} +{:.2f} -{:.2f} Msun'.format(q[1],q[2]-q[1],q[1]-q[0]))

In [ ]:
# i
sig = df3['M2_Msun'].std()
i = df3["M2_Msun"].var()
print(f'i = {i:.2f} off by: {i * sig**-1:.2f}')
# ii
ii = q[2]-q[0]
print(f'ii = {ii:.2f} off by: {ii * sig**-1:.2f}')
# iii
iii = sig/np.sqrt(8000)
print(f'iii = {iii:.2f} off by: {iii * sig**-1:.2f}')


i is just the variance or $\sigma^2$ and has units of of M_sun^2. It is 1.16x larger than the $1\sigma$ error bar

ii is the full width of the 68% range and should be around $2\sigma$. It has units M_sun and is actually 1.95 larger than $1\sigma$

iii is the standard error of the sample mean, It answers how well the posterior mean has been sampled from the available data. It has units M_sun and is 0.01x smaller that the true $1\sigma$

In [ ]:
df3['M2_Msun'].plot.hist()

**ANSWER:**

Invesitgatig the spead of the data it appears to be a positively skewed distribution. Therefore I do not think that a symetric error bound would be appropiate for this data. What I would report is the 16th-84% error bars because it reprients the true spread of the data around the median. The upper and lower bounds are found by finding the difference between the median and the respective percentage. $4.89^{+1.28}_{-0.99}$ Msun

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
df4 = load('p1e_slope_intercept_samples')
slope = df4['slope'].to_numpy()
intercept = df4['intercept'].to_numpy()

rho = np.corrcoef(slope,intercept)[0,1]
sig_slope_marginal = np.std(slope,ddof=1)
sig_slope_classmate = sig_slope_marginal*np.sqrt(1-rho**2)

print("Marginal slope sigma = {:.4f}".format(sig_slope_marginal))
print("Classmate's conditional slope sigma = {:.4f}".format(sig_slope_classmate))
print('rho = {:.3f}, conditional/marginal sigma ratio = {:.3f}'.format(rho,sig_slope_classmate/sig_slope_marginal))

The ratio depends on $sqrt(1-rho**2)$

In [ ]:
'''1e.2: Marginal versus conditional uncertainty'''
# --------------------------------------------------
# Joint posterior and fixed-intercept slice
# --------------------------------------------------
intercept_med = np.median(intercept)
width = 0.05*np.std(intercept,ddof=1)
fixed = np.abs(intercept-intercept_med)<width

fig,ax = plt.subplots(1,2,figsize=(10,4))

ax[0].scatter(intercept,slope,s=5,alpha=0.2,label='Joint samples')
ax[0].axvline(intercept_med,color='C1',ls='--',label='Fixed intercept')
ax[0].axvspan(intercept_med-width,intercept_med+width,color='C1',alpha=0.15)
ax[0].set_xlabel('Intercept')
ax[0].set_ylabel('Slope')
ax[0].set_title('Joint posterior, rho = {:.3f}'.format(rho))
ax[0].legend()

ax[1].hist(slope,bins=35,density=True,alpha=0.6,label='Marginal slope')
ax[1].hist(slope[fixed],bins=20,density=True,alpha=0.6,label='Intercept fixed near median')
ax[1].set_xlabel('Slope')
ax[1].set_ylabel('Probability density')
ax[1].set_title('Marginal versus conditional')
ax[1].legend()

plt.tight_layout()
plt.show()



**ANSWER:**

The classmates answer assumes that the intercept is known. I would only quote their value if that was true.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
'''2a: Distance modulus and Hubble residuals'''
# --------------------------------------------------
# Cosmological distance modulus
# --------------------------------------------------
from astropy.cosmology import FlatLambdaCDM

p2 = load('p2_pantheon_sne')

z = p2['z'].to_numpy()
mB = p2['mB'].to_numpy()
x1 = p2['x1'].to_numpy()
color = p2['c'].to_numpy()
host_logmass = p2['host_logmass'].to_numpy()

cosmo = FlatLambdaCDM(H0=70,Om0=0.3)
mu = cosmo.distmod(z).value
hubble_resid = mB-mu

if np.any(~np.isfinite(np.column_stack((z,mB,x1,color,host_logmass,mu,hubble_resid)))):
    raise ValueError('Part 2a contains non-finite values.')
if np.any(z<=0):
    raise ValueError('Distance modulus requires positive redshifts.')

c_kms = 299792.458
sigma_mu_pec = (5/np.log(10))*(250/(c_kms*z))

fig,ax = plt.subplots(1,3,figsize=(13,4),sharey=True)

ax[0].scatter(x1,hubble_resid,s=20,alpha=0.7)
ax[0].set_xlabel(r'$x_1$')
ax[0].set_ylabel(r'$m_B-\mu(z)$ (mag)')
ax[0].set_title('Residual versus stretch')

ax[1].scatter(color,hubble_resid,s=20,alpha=0.7)
ax[1].set_xlabel(r'$c$')
ax[1].set_title('Residual versus color')

ax[2].scatter(host_logmass,hubble_resid,s=20,alpha=0.7)
ax[2].axvline(10,color='C1',ls='--',label=r'$\log_{10}(M_*/M_\odot)=10$')
ax[2].set_xlabel(r'Host $\log_{10}(M_*/M_\odot)$')
ax[2].set_title('Residual versus host mass')
ax[2].legend()

plt.tight_layout()
plt.show()

print('N supernovae = {}'.format(len(p2)))
print('Redshift range = {:.4f} to {:.4f}'.format(np.min(z),np.max(z)))
print('mu(z) range = {:.3f} to {:.3f} mag'.format(np.min(mu),np.max(mu)))
print('Median mB-mu(z) = {:.3f} mag'.format(np.median(hubble_resid)))

The x1 trend is negative and the c trend is positive. Both of these trends are apperant in the tripp relation, the data is behaving as expected.

In [ ]:
plt.scatter(host_logmass,x1)

plt.title('x1 Against Host Mass');plt.xlabel(r'Host $\log_{10}(M_*/M_\odot)$');plt.ylabel('x1')

$\gamma$ won't be reprisented in this plot because it doesn't effect the relationship between x1 and host mass. Effects from $\gamma$ will appear as on offset in standardized brightness after accounting for stretch and color.

**ANSWER:**

I can only clearly see one contaminant and it has a Hubble residual of around -12.1 mag. The other 1-3 contaminants are not ovious in my data.

Both $\alpha$ and $\beta$ have units of mag because x1 and c are unitless.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

# 2b: your work here

$$
\mathbf{v}=
\begin{pmatrix}
1\\
\alpha\\
-\beta
\end{pmatrix},
\qquad
\mathbf{C}_i=
\begin{pmatrix}
\sigma_{m_B,i}^2 & \mathrm{Cov}(m_B,x_1)_i & \mathrm{Cov}(m_B,c)_i\\
\mathrm{Cov}(m_B,x_1)_i & \sigma_{x_1,i}^2 & \mathrm{Cov}(x_1,c)_i\\
\mathrm{Cov}(m_B,c)_i & \mathrm{Cov}(x_1,c)_i & \sigma_{c,i}^2
\end{pmatrix}
$$

$$
\mathbf{v}^{T}\mathbf{C}_i\mathbf{v}
=
\begin{pmatrix}
1 & \alpha & -\beta
\end{pmatrix}
\begin{pmatrix}
\sigma_{m_B,i}^2 & \mathrm{Cov}(m_B,x_1)_i & \mathrm{Cov}(m_B,c)_i\\
\mathrm{Cov}(m_B,x_1)_i & \sigma_{x_1,i}^2 & \mathrm{Cov}(x_1,c)_i\\
\mathrm{Cov}(m_B,c)_i & \mathrm{Cov}(x_1,c)_i & \sigma_{c,i}^2
\end{pmatrix}
\begin{pmatrix}
1\\
\alpha\\
-\beta
\end{pmatrix}.
$$


$$
\boxed{
\mathbf{v}^{T}\mathbf{C}_i\mathbf{v}
=
\sigma_{m_B,i}^{2}
+\alpha^{2}\sigma_{x_1,i}^{2}
+\beta^{2}\sigma_{c,i}^{2}
+2\alpha\,\mathrm{Cov}(m_B,x_1)_i
-2\beta\,\mathrm{Cov}(m_B,c)_i
-2\alpha\beta\,\mathrm{Cov}(x_1,c)_i
}
$$

Each off diagonal term in $C_i$ reprisents the covariance between pairs of SALT2 fitted quantities. $m_B, x_1,c$ all come from the same light curve fit so they do not have independed error, if one quantity shifts they all might. These terms ensure that I don't treat the fitted quantities as independent which could lead to an incorrect uncertainty.

In [ ]:
'''2b.2: Log-likelihood'''
# --------------------------------------------------
# Tripp relation with host-mass step
# --------------------------------------------------
def log_likelihood(p):
    alpha,beta,M_B,gamma,log_sig_int = p
    sig_int = np.exp(log_sig_int)

    high_mass = (host_logmass>=10).astype(float)

    resid = mB-mu-M_B+alpha*x1-beta*color-gamma*high_mass

    vTCv = (p2['mB_err'].to_numpy()**2
            +alpha**2*p2['x1_err'].to_numpy()**2
            +beta**2*p2['c_err'].to_numpy()**2
            +2*alpha*p2['cov_mB_x1'].to_numpy()
            -2*beta*p2['cov_mB_c'].to_numpy()
            -2*alpha*beta*p2['cov_x1_c'].to_numpy())

    sig2 = vTCv+sigma_mu_pec**2+sig_int**2

    if np.any(~np.isfinite(sig2)) or np.any(sig2<=0):
        return -np.inf

    return -0.5*np.sum(resid**2/sig2+np.log(sig2)+np.log(2*np.pi))


###### Priors 

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$

$\alpha$ = (0.0,0.5)

The lower bound for $\alpha$ requires it no be non negative, this is consistent with broader, slower-declining Type 1a supernovae. The upper bound allows a much wider range than expected while still excluding extreem conditions that could compesate for contaminants or model errors

$\beta$ = (0.0,6.0)

This prior requires a nonegative color correction, a negative value would imply redder supernovae are intrinsically brighter which is incorrect. The upper bound is intentionally broad but still prevents extreem color corrections from dominating the fit

M = (-20.5,-18.0)

This range broadly brackets the expected standardized Type 1a absolute magnitude and should allow data to determine the intercept without permitting values inconsistent with the intended source population.

$\gamma$ = (-0.5,0.5)

This prior for $\gamma$ allows either dirrection for the most mass step and to avoid building in an expected sign. The interval is intended to be broad based on the anticipated effect.

$\sigma_{\rm int}$ = (0.001,0.5)

The lower bound is to approximate zero while remaining compatble with the natural log and the upper bound keeps the model from hiding a poor fit by continually increasing sigma.

I will be sampling $\ln\sigma_{\rm int}$ and transform it inside the likelyhood. My reasoning for this is to maintain numerical stability. That being said, my prior is uniform in $\sigma_{\rm int}$

To handle the contaminats I will perform a 4-sigma standardize residual criterion 

In [ ]:
'''2b: Simple 4-sigma contaminant cut'''
# --------------------------------------------------
# Preliminary Tripp fit and studentized residuals
# --------------------------------------------------
from astropy.cosmology import FlatLambdaCDM

p2_all = p2.copy().reset_index(drop=True)

z = p2_all['z'].to_numpy()
mB = p2_all['mB'].to_numpy()
x1 = p2_all['x1'].to_numpy()
color = p2_all['c'].to_numpy()
host_logmass = p2_all['host_logmass'].to_numpy()
high_mass = (host_logmass>=10).astype(float)

cosmo = FlatLambdaCDM(H0=70,Om0=0.3)
mu = cosmo.distmod(z).value

# mB-mu = M_B-alpha*x1+beta*c+gamma*high_mass
y = mB-mu
X = np.column_stack((np.ones(len(p2_all)),-x1,color,high_mass))

theta = np.linalg.lstsq(X,y,rcond=None)[0]
model = X@theta
resid = y-model

n,k = X.shape
s2 = resid@resid/(n-k)
leverage = np.sum(X*(X@np.linalg.inv(X.T@X)),axis=1)
studentized_resid = resid/np.sqrt(s2*(1-leverage))

cut = 4.0
contaminant = np.abs(studentized_resid)>cut
p2_clean = p2_all.loc[~contaminant].copy().reset_index(drop=True)

flagged = p2_all.loc[contaminant,['name','z','mB','x1','c','host_logmass']].copy()
flagged['resid_mag'] = resid[contaminant]
flagged['studentized_resid'] = studentized_resid[contaminant]
flagged['abs_studentized_resid'] = np.abs(studentized_resid[contaminant])
flagged = flagged.sort_values('abs_studentized_resid',ascending=False)

print('Criterion: abs(studentized residual) > {:.1f}'.format(cut))
print('Original sources = {}'.format(len(p2_all)))
print('Flagged sources = {}'.format(np.sum(contaminant)))
print('Clean sources = {}'.format(len(p2_clean)))

if len(flagged)>0:
    print('\nSources removed:')
    print(flagged.to_string(index=False))
else:
    print('\nNo sources were removed.')

if not (2<=len(flagged)<=4):
    print('\nWARNING: The prompt states that 2 to 4 contaminants are present.')
    print('Review this preliminary cut before using p2_clean.')

plt.figure(figsize=(7,4))
plt.axhline(0,color='k',lw=1)
plt.axhline(cut,color='C1',ls='--')
plt.axhline(-cut,color='C1',ls='--')
plt.scatter(z,studentized_resid,s=20,alpha=0.7)
plt.scatter(z[contaminant],studentized_resid[contaminant],
            s=60,facecolors='none',edgecolors='C3',label='Flagged')
plt.xlabel('Redshift')
plt.ylabel('Studentized residual')
if np.any(contaminant):
    plt.legend()
plt.tight_layout()
plt.show()

p2 = p2_clean.copy()

z = p2['z'].to_numpy()

mB = p2['mB'].to_numpy()
x1 = p2['x1'].to_numpy()
color = p2['c'].to_numpy()
host_logmass = p2['host_logmass'].to_numpy()

mu = cosmo.distmod(z).value

c_kms = 299792.458
sigma_mu_pec = (5/np.log(10))*(250/(c_kms*z))

**ANSWER:**

To remove the contaminants I flagged sources with extreemly large residuals. Looking at the residual plot it is pretty clear how these sources do not fit with the rets of the data. I am removing from the data fram and then continuing onwards with the cleaned dataset. This cut could cost me some real data but assuming the data was distributed randomly with no contaminents this cut should remove only extremely unusual points under the assumed noise model. If these values were real I could be ommiting some misunderstood phenomena.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
'''2c: MAP'''
# --------------------------------------------------
# MAP fit
# --------------------------------------------------
import emcee
import arviz as az

print('len(p2) =',len(p2))

print('mB =',mB.shape)
print('x1 =',x1.shape)
print('color =',color.shape)
print('host_logmass =',host_logmass.shape)
print('mu =',mu.shape)
print('sigma_mu_pec =',sigma_mu_pec.shape)

def log_prior(p):
    alpha,beta,M_B,gamma,log_sig_int = p
    sig_int = np.exp(log_sig_int)

    if not (0.0 < alpha < 0.5):
        return -np.inf
    if not (0.0 < beta < 6.0):
        return -np.inf
    if not (-20.5 < M_B < -18.0):
        return -np.inf
    if not (-0.5 < gamma < 0.5):
        return -np.inf
    if not (0.001 < sig_int < 0.5):
        return -np.inf

    return 0.0

def log_posterior(p):
    lp = log_prior(p)

    if not np.isfinite(lp):
        return -np.inf

    return lp + log_likelihood(p)

theta0 = np.array([0.14,3.0,-19.3,0.0,np.log(0.1)])

result = minimize(lambda p: -log_posterior(p),theta0,method='Nelder-Mead',options=dict(maxiter=20000,xatol=1e-6,fatol=1e-6))

best_p = result.x

print('MAP fit:')
print('alpha = {:.4f}'.format(best_p[0]))
print('beta = {:.4f}'.format(best_p[1]))
print('M_B = {:.4f}'.format(best_p[2]))
print('gamma = {:.4f}'.format(best_p[3]))
print('sig_int = {:.4f}'.format(np.exp(best_p[4])))


In [ ]:
'''2c: MCMC'''
# --------------------------------------------------
# Four independent chains
# --------------------------------------------------
nchains = 4
nwalkers = 32
ndim = len(best_p)
nsteps = 10000

chains = []
taus = []

for chain_id in range(nchains):

    rng = np.random.default_rng(10132026+chain_id)

    scale = np.array([
        0.05,          # alpha
        0.5,           # beta
        0.3,           # M_B
        0.05,          # gamma
        0.5            # log_sig_int
    ])

    offsets = [
        np.array([ 0.03, 0.30, 0.15, 0.03, 0.20]),
        np.array([-0.03,-0.30,-0.15,-0.03,-0.20]),
        np.array([ 0.02,-0.25, 0.10,-0.02, 0.10]),
        np.array([-0.02, 0.25,-0.10, 0.02,-0.10])
    ]
    
    pos = best_p + offsets[chain_id] + rng.normal(
        0,
        scale,
        size=(nwalkers,ndim)
    )

    sampler = emcee.EnsembleSampler(
        nwalkers,
        ndim,
        log_posterior
    )

    sampler.run_mcmc(pos,nsteps,progress=False)

    tau = sampler.get_autocorr_time()

    print('\nChain {}'.format(chain_id+1))

    for name,value in zip(
        ['alpha','beta','M_B','gamma','log_sig_int'],
        tau
    ):
        print('{}: tau = {:.1f}'.format(name,value))

    print('Chain/tau = {:.1f}'.format(
        nsteps/np.max(tau)
    ))

    chains.append(sampler)
    taus.append(tau)

taus = np.array(taus)

# --------------------------------------------------
# Burn-in and thinning
# --------------------------------------------------
tau_max = np.max(taus)

burnin = int(3*tau_max)
thin = max(1,int(tau_max/2))

print('\nBurn-in = {}'.format(burnin))
print('Thin = {}'.format(thin))
print('nsteps > 50*tau ? {}'.format(
    nsteps > 50*tau_max
))

# --------------------------------------------------
# Plot post-burn-in chains
# --------------------------------------------------
names = ['alpha','beta','M_B','gamma','log_sig_int']

for chain_id,sampler in enumerate(chains):

    chain = sampler.get_chain(
        discard=burnin,
        thin=thin,
        flat=False
    )

    fig,axes = plt.subplots(
        ndim,
        1,
        figsize=(10,8),
        sharex=True
    )

    for i,name in enumerate(names):
        axes[i].plot(chain[:,:,i],alpha=0.25)
        axes[i].set_ylabel(name)

    axes[-1].set_xlabel('Step')

    plt.suptitle(
        'Chain {} After Burn-in and Thinning'.format(
            chain_id+1
        )
    )

    plt.tight_layout()
    plt.show()

# --------------------------------------------------
# Combine posterior samples
# --------------------------------------------------
samples = np.concatenate([
    sampler.get_chain(
        discard=burnin,
        thin=thin,
        flat=True
    )
    for sampler in chains
])

print('Posterior samples = {}'.format(len(samples)))

In [ ]:
posterior = {}

for i,name in enumerate(['alpha','beta','M_B','gamma','log_sig_int']):
    posterior[name] = np.array([
        sampler.get_chain(discard=burnin,thin=thin)[:,:,i].T
        for sampler in chains
    ])[:,0,:]


idata = az.from_dict({'posterior':posterior})

summary = az.summary(idata)

print(summary[['ess_bulk','r_hat']])

R_hat is a to see if all of my chains converged on the same statistic. Having all of them equal to 1 after giving each chain a different starting point is a good sign that I have converged on the correct answer. If I had a parameter with R_hat > 1.01 the code would have run longer, because the four chains would be sampling different regions.

ESS is the effective sample size because not every sample in emcee is independent from the surrounding ones. If ESS was only a few hundred samples or less (ESS < 400) the code would have run longer because posterior summaries and uncertainty estimates would be dominated by MC noise.

In [ ]:
'''Corner plot'''
import corner
alpha_s = samples[:,0]
beta_s = samples[:,1]
M_B_s = samples[:,2]
gamma_s = samples[:,3]
log_sig_int_s = samples[:,4]

sig_int_s = np.exp(log_sig_int_s)

physical_samples = np.column_stack((alpha_s,beta_s,M_B_s,gamma_s,sig_int_s))

labels = [r'$\alpha$',r'$\beta$',r'$M_B$',r'$\gamma$',r'$\sigma_{\rm int}$']

fig = corner.corner(physical_samples,labels=labels,show_titles=True,quantiles=[0.16,0.5,0.84],
                    title_fmt='.2f',title_kwargs={'fontsize':10})
plt.show()

The only parameters that are strongly correlated with eachother are $\gamma$ and $M_B$. This is to be expected because $M_B$ and $\gamma$ both shift the predicted sipernova brightness and can partially compensate for one another.

In [ ]:
'''Posterior results'''

alpha_q = np.percentile(alpha_s,[16,50,84])
beta_q = np.percentile(beta_s,[16,50,84])
M_B_q = np.percentile(M_B_s,[16,50,84])
gamma_q = np.percentile(gamma_s,[16,50,84])
sig_int_q = np.percentile(sig_int_s,[16,50,84])


def report(name,q,unit='',decimals=2):
    value = q[1]
    plus = q[2]-q[1]
    minus = q[1]-q[0]
    print(r'{} = {:.{}f} +{:.{}f} -{:.{}f} {}'.format(name,value,decimals,plus,decimals,minus,decimals,unit).rstrip())




print('Posterior results:')
report('α',alpha_q,'mag',2)
report('β',beta_q,'mag',2)
report('M_B',M_B_q,'mag',3)
report('γ',gamma_q,'mag',2)
report('σ_int',sig_int_q,'mag',2)

In [ ]:
'''2c.4: Gamma with M_B fixed'''
# --------------------------------------------------
# Compare conditional and marginal gamma uncertainty
# --------------------------------------------------

# Marginal uncertainty from MCMC
gamma_q = np.percentile(gamma_s,[16,50,84])
sig_gamma_marginal = 0.5*((gamma_q[2]-gamma_q[1])+(gamma_q[1]-gamma_q[0]))

# Fix M_B at MAP value
M_B_fixed = best_p[2]

def log_likelihood_fixed_M(p):
    alpha,beta,gamma,log_sig_int = p

    sig_int = np.exp(log_sig_int)
    high_mass = (host_logmass>=10).astype(float)

    resid = mB-mu-M_B_fixed+alpha*x1-beta*color-gamma*high_mass

    vTCv = (
        p2['mB_err'].to_numpy()**2
        +alpha**2*p2['x1_err'].to_numpy()**2
        +beta**2*p2['c_err'].to_numpy()**2
        +2*alpha*p2['cov_mB_x1'].to_numpy()
        -2*beta*p2['cov_mB_c'].to_numpy()
        -2*alpha*beta*p2['cov_x1_c'].to_numpy()
    )

    sig2 = vTCv+sigma_mu_pec**2+sig_int**2

    if np.any(~np.isfinite(sig2)) or np.any(sig2<=0):
        return -np.inf

    return -0.5*np.sum(
        resid**2/sig2+np.log(sig2)+np.log(2*np.pi)
    )

def log_prior_fixed_M(p):
    alpha,beta,gamma,log_sig_int = p

    sig_int = np.exp(log_sig_int)

    if not (0.0<alpha<0.5):
        return -np.inf
    if not (0.0<beta<6.0):
        return -np.inf
    if not (-0.5<gamma<0.5):
        return -np.inf
    if not (0.001<sig_int<0.5):
        return -np.inf

    return 0.0

def log_posterior_fixed_M(p):
    lp = log_prior_fixed_M(p)

    if not np.isfinite(lp):
        return -np.inf

    return lp+log_likelihood_fixed_M(p)

theta0_fixed = np.array([
    best_p[0],
    best_p[1],
    best_p[3],
    best_p[4]
])

result_fixed = minimize(
    lambda p:-log_posterior_fixed_M(p),
    theta0_fixed,
    method='Nelder-Mead',
    options=dict(
        maxiter=20000,
        xatol=1e-6,
        fatol=1e-6
    )
)

alpha_fix,beta_fix,gamma_fix,log_sig_fix = result_fixed.x

# Numerical Hessian element for gamma
eps = 1e-4

p0 = result_fixed.x.copy()

p_plus = p0.copy()
p_plus[2] += eps

p_minus = p0.copy()
p_minus[2] -= eps

H_gamma = (
    -log_posterior_fixed_M(p_plus)
    +2*log_posterior_fixed_M(p0)
    -log_posterior_fixed_M(p_minus)
)/(eps**2)

sig_gamma_fixed = np.sqrt(1/H_gamma)

print('gamma MAP (fixed M_B) = {:.4f}'.format(gamma_fix))
print('Conditional sigma_gamma = {:.4f} mag'.format(sig_gamma_fixed))
print('Marginal sigma_gamma = {:.4f} mag'.format(sig_gamma_marginal))
print('Ratio conditional/marginal = {:.3f}'.format(
    sig_gamma_fixed/sig_gamma_marginal
))

**ANSWER:**

Fixing $M_B$ as the MAP value reduced $\sigma_\gamma$ from 0.0192 to 0.0120. $\sigma_\gamma$ fixed is only 62.4% the size of the marginal error. This is an effect of $M_B$ being correlated with $\gamma$. The uncertainty in $M_B$ propigates to $\gamma$ and once $M_B$ is fixed that uncertainty is artificially smaller and is only appropiate if $M_B$ is known.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here
'''2d: Laplace Bayes factor'''
# --------------------------------------------------
# Model M0 : gamma fixed to zero
# --------------------------------------------------

def log_likelihood_M0(p):
    alpha,beta,M_B,log_sig_int = p

    sig_int = np.exp(log_sig_int)
    high_mass = (host_logmass>=10).astype(float)

    resid = mB-mu-M_B+alpha*x1-beta*color

    vTCv = (p2['mB_err'].to_numpy()**2+alpha**2*p2['x1_err'].to_numpy()**2+beta**2*p2['c_err'].to_numpy()**2+2*alpha*p2['cov_mB_x1'].to_numpy()-2*beta*p2['cov_mB_c'].to_numpy()-2*alpha*beta*p2['cov_x1_c'].to_numpy())

    sig2 = vTCv+sigma_mu_pec**2+sig_int**2

    if np.any(sig2<=0):
        return -np.inf
    return -0.5*np.sum(resid**2/sig2+np.log(sig2)+np.log(2*np.pi))

def log_prior_M0(p):
    alpha,beta,M_B,log_sig_int = p
    sig_int = np.exp(log_sig_int)
    if not (0<alpha<0.5):
        return -np.inf
    if not (0<beta<6):
        return -np.inf
    if not (-20.5<M_B<-18):
        return -np.inf
    if not (0.001<sig_int<0.5):
        return -np.inf
    return 0.

def log_posterior_M0(p):
    lp = log_prior_M0(p)
    if not np.isfinite(lp):
        return -np.inf
    return lp+log_likelihood_M0(p)

theta0_M0 = np.array([best_p[0],best_p[1],best_p[2],best_p[4]])

result_M0 = minimize(lambda p:-log_posterior_M0(p),theta0_M0,method='Nelder-Mead',options=dict(maxiter=20000,xatol=1e-6,fatol=1e-6))

In [ ]:
def hessian(fun,x,eps=1e-5):
    n = len(x)
    H = np.zeros((n,n))
    for i in range(n):
        for j in range(n):

            xp = x.copy()
            xm = x.copy()
            xpp = x.copy()
            xmm = x.copy()
            xp[i] += eps
            xp[j] += eps
            xm[i] += eps
            xm[j] -= eps
            xpp[i] -= eps
            xpp[j] += eps
            xmm[i] -= eps
            xmm[j] -= eps
            H[i,j] = (fun(xp)-fun(xm)-fun(xpp)+fun(xmm)
            )/(4*eps**2)
    return H

In [ ]:
H1 = hessian(lambda p:-log_posterior(p),result.x)

H0 = hessian(lambda p:-log_posterior_M0(p),result_M0.x)

C1 = np.linalg.inv(H1)
C0 = np.linalg.inv(H0)

k1 = len(result.x)
k0 = len(result_M0.x)

logL1 = log_likelihood(result.x)
logL0 = log_likelihood_M0(result_M0.x)

prior_widths_M1 = (0.5*6.0*2.5*1.0*np.log(0.5/0.001))

prior_widths_M0 = (0.5*6.0*2.5*np.log(0.5/0.001))

logpi1 = -np.log(prior_widths_M1)
logpi0 = -np.log(prior_widths_M0)

logZ1 = (logL1+logpi1+(k1/2)*np.log(2*np.pi)+0.5*np.log(np.linalg.det(C1)))

logZ0 = (logL0+logpi0+(k0/2)*np.log(2*np.pi)+0.5*np.log(np.linalg.det(C0)))

two_ln_B10 = 2*(logZ1-logZ0)

print('2ln(B10) = {:.3f}'.format(two_ln_B10))

In [ ]:
logpi1_wide = -np.log(
    prior_widths_M1*2
)

logZ1_wide = (
    logL1
    +logpi1_wide
    +(k1/2)*np.log(2*np.pi)
    +0.5*np.log(np.linalg.det(C1))
)

two_ln_B10_wide = 2*(logZ1_wide-logZ0)

print(
    '2ln(B10) with doubled gamma prior = {:.3f}'
    .format(two_ln_B10_wide)
)

When the prior for $\gamma$ is dubled the value for $2ln(B_{1.0})$ decreased by ~ a factor of 2. 

In [ ]:
'''2d: BIC comparison'''
# --------------------------------------------------
# BIC0 - BIC1
# --------------------------------------------------
N = len(p2)

logL1 = log_likelihood(result.x)
logL0 = log_likelihood_M0(result_M0.x)

k1 = 5
k0 = 4

BIC1 = k1*np.log(N)-2*logL1
BIC0 = k0*np.log(N)-2*logL0

delta_BIC = BIC0-BIC1

print('BIC0 = {:.3f}'.format(BIC0))
print('BIC1 = {:.3f}'.format(BIC1))
print('BIC0 - BIC1 = {:.3f}'.format(delta_BIC))

In [ ]:
print('gamma = {:.4f} +/- {:.4f} mag'.format(
    gamma_q[1],
    sig_gamma_marginal
))
print('Published step (-0.06 mag) corresponds to {:.2f} sigma'.format(
    0.06/sig_gamma_marginal
))

**ANSWER:**

The Laplace approximation between a model without $\gamma$ one with my prior showed week evidence that favored a model without $\gamma$. When the prior for $\gamma$ is doubled the approximation gives positive evidence in favor of the $\gamma$-less model. The BIC difference slightly favors a model without $\gamma$. Taking all of this into account I do not find any strong evidence that a $\gamma$ term is required for this dataset. My fitted range for $\gamma$ is consistent in sign and magnitude as the published falue but the published value corresponds to ~ $3.2\sigma$. In principle this data set could detect a step of that size.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.


#### Coverage test

Using a similar test as in lab02, I will run my pipeline on many different sets of test data by simulating fake magnitudes at the actual redshifts, x1 values, colors, host masses, and covariences from my cleaned dataset. Ideally I will find that the 1$\sigma$ error bounds cover the truth ~68% of the time, if that criteria is met my pipeline passes this test.

In [ ]:
# 2e: your work here
'''2e: Coverage test'''
# --------------------------------------------------
# Monte Carlo verification
# --------------------------------------------------
rng = np.random.default_rng(10142026)

truth = dict(alpha=0.14,
    beta=3.10,
    M_B=-19.30,
    gamma=-0.06,
    sig_int=0.10
)

nmc = 1000

results = []

for seed in range(nmc):

    rng = np.random.default_rng(seed)

    alpha_t = truth['alpha']
    beta_t = truth['beta']
    M_B_t = truth['M_B']
    gamma_t = truth['gamma']
    sig_int_t = truth['sig_int']

    high_mass = (host_logmass>=10).astype(float)

    model = (
        mu
        + M_B_t
        - alpha_t*x1
        + beta_t*color
        + gamma_t*high_mass
    )

    vTCv = (
        p2['mB_err'].to_numpy()**2
        + alpha_t**2*p2['x1_err'].to_numpy()**2
        + beta_t**2*p2['c_err'].to_numpy()**2
        + 2*alpha_t*p2['cov_mB_x1'].to_numpy()
        - 2*beta_t*p2['cov_mB_c'].to_numpy()
        - 2*alpha_t*beta_t*p2['cov_x1_c'].to_numpy()
    )

    sig_tot = np.sqrt(
        vTCv
        + sigma_mu_pec**2
        + sig_int_t**2
    )

    mB_sim = model + rng.normal(0,sig_tot)

    mB_old = mB.copy()
    mB = mB_sim

    theta0 = np.array([
        alpha_t,
        beta_t,
        M_B_t,
        gamma_t,
        np.log(sig_int_t)
    ])

    fit = minimize(
        lambda p:-log_posterior(p),
        theta0,
        method='Nelder-Mead',
        options=dict(
            maxiter=20000,
            xatol=1e-6,
            fatol=1e-6
        )
    )

    if not fit.success:
        continue

    p0 = fit.x.copy()

    H = np.zeros((5,5))
    eps = 1e-4

    for i in range(5):
        for j in range(5):

            ei = np.zeros(5)
            ej = np.zeros(5)

            ei[i] = eps
            ej[j] = eps

            H[i,j] = (
                (-log_posterior(p0+ei+ej))
                -(-log_posterior(p0+ei-ej))
                -(-log_posterior(p0-ei+ej))
                +(-log_posterior(p0-ei-ej))
            )/(4*eps**2)

    try:
        cov = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        continue

    sigma = np.sqrt(np.diag(cov))

    fit_alpha,fit_beta,fit_MB,fit_gamma,fit_logsig = fit.x

    fit_sigint = np.exp(fit_logsig)
    sigma_sigint = fit_sigint*sigma[4]

    results.append(dict(
        alpha=np.abs(fit_alpha-alpha_t)<=sigma[0],
        beta=np.abs(fit_beta-beta_t)<=sigma[1],
        M_B=np.abs(fit_MB-M_B_t)<=sigma[2],
        gamma=np.abs(fit_gamma-gamma_t)<=sigma[3],
        sig_int=np.abs(fit_sigint-sig_int_t)<=sigma_sigint
    ))

    mB = mB_old

coverage = pd.DataFrame(results)

summary = pd.DataFrame({
    'coverage':coverage.mean(),
    'failures':len(coverage)-coverage.sum()
})

print(summary)

**ANSWER:**

After testing on 1000 different sets of data I find that each parameter is covered ~68% of the time. This gives me confidence that my pipeline is working is a way that it can accurately retriev the true values according to the error bounds that I define.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
p3 = load('p3_wd_masses')

p3['mass_Msun'].plot.hist(bins = 30)

##### Identify

1. The AI performs a $3\sigma$ clip on the observed WD mass before fitting with a mixture model. 
2. The AI solution ignores the per star mass errors in its mixture model
3. The AI validates the data on only one synthetic dataset instead of checking over many

In [ ]:
'''Part 3: Demonstrate why the 3-sigma cut is problematic'''
# --------------------------------------------------
# Reproduce AI clipping
# --------------------------------------------------
mass = p3['mass_Msun'].to_numpy()

mu = np.mean(mass)
sig = np.std(mass,ddof=1)

keep = np.abs(mass-mu) < 3*sig
removed = ~keep

print('Original sample size = {}'.format(len(mass)))
print('Removed by 3-sigma cut = {}'.format(np.sum(removed)))
print('Fraction removed = {:.2%}'.format(np.mean(removed)))

# --------------------------------------------------
# Quantify effect on high-mass population
# --------------------------------------------------
high_mass = mass > 0.8

print('\nHigh-mass stars (>0.8 Msun)')
print('Before clipping = {}'.format(np.sum(high_mass)))
print('After clipping = {}'.format(np.sum(high_mass & keep)))
print('Removed high-mass stars = {}'.format(np.sum(high_mass & removed)))

# --------------------------------------------------
# Distribution changes
# --------------------------------------------------
print('\nDistribution summary')
print('Mean before = {:.3f}'.format(np.mean(mass)))
print('Mean after  = {:.3f}'.format(np.mean(mass[keep])))

print('Std before  = {:.3f}'.format(np.std(mass,ddof=1)))
print('Std after   = {:.3f}'.format(np.std(mass[keep],ddof=1)))

# --------------------------------------------------
# Plot
# --------------------------------------------------
plt.figure(figsize=(8,5))

plt.hist(
    mass,
    bins=40,
    alpha=0.5,
    density=True,
    label='All white dwarfs'
)

plt.hist(
    mass[keep],
    bins=40,
    alpha=0.5,
    density=True,
    label='After AI 3σ clip'
)

plt.xlabel(r'Mass ($M_\odot$)')
plt.ylabel('Density')
plt.title('Effect of AI 3σ Clipping')
plt.legend()

plt.show()

# --------------------------------------------------
# Show removed objects
# --------------------------------------------------
removed_df = p3.loc[removed,[
    'name',
    'mass_Msun',
    'mass_err_Msun',
    'teff_K'
]]

print('\nRemoved white dwarfs:')
print(removed_df.sort_values('mass_Msun').to_string(index=False))

In [ ]:
'''Part 3: Ignoring mass errors'''
# --------------------------------------------------
# Compare mixture widths with and without mass errors
# --------------------------------------------------
from scipy.stats import norm
from scipy.optimize import minimize

mass = p3['mass_Msun'].to_numpy()
mass_err = p3['mass_err_Msun'].to_numpy()

def nll_noerr(p):
    mu1,sig1,mu2,sig2,f = p

    if sig1<=0 or sig2<=0 or f<=0 or f>=1:
        return np.inf

    model = (
        f*norm.pdf(mass,mu1,sig1)
        +(1-f)*norm.pdf(mass,mu2,sig2)
    )

    return -np.sum(np.log(model+1e-300))

def nll_witherr(p):
    mu1,sig1,mu2,sig2,f = p

    if sig1<=0 or sig2<=0 or f<=0 or f>=1:
        return np.inf

    sig1_eff = np.sqrt(sig1**2+mass_err**2)
    sig2_eff = np.sqrt(sig2**2+mass_err**2)

    model = (
        f*norm.pdf(mass,mu1,sig1_eff)
        +(1-f)*norm.pdf(mass,mu2,sig2_eff)
    )

    return -np.sum(np.log(model+1e-300))

p0 = [0.60,0.05,0.85,0.05,0.7]

fit_noerr = minimize(nll_noerr,p0,method='Nelder-Mead')
fit_witherr = minimize(nll_witherr,p0,method='Nelder-Mead')

mu1_n,sig1_n,mu2_n,sig2_n,f_n = fit_noerr.x
mu1_e,sig1_e,mu2_e,sig2_e,f_e = fit_witherr.x

print('Ignoring mass errors')
print('mu1 = {:.3f}  sigma1 = {:.3f}'.format(mu1_n,sig1_n))
print('mu2 = {:.3f}  sigma2 = {:.3f}'.format(mu2_n,sig2_n))

print('\nIncluding mass errors')
print('mu1 = {:.3f}  sigma1 = {:.3f}'.format(mu1_e,sig1_e))
print('mu2 = {:.3f}  sigma2 = {:.3f}'.format(mu2_e,sig2_e))

print('\nWidth ratios')
print('sigma1(with err)/sigma1(no err) = {:.3f}'.format(sig1_e/sig1_n))
print('sigma2(with err)/sigma2(no err) = {:.3f}'.format(sig2_e/sig2_n))

In [ ]:
'''Single realization is not coverage'''
# --------------------------------------------------
# Demonstrate why one closure test is insufficient
# --------------------------------------------------
rng = np.random.default_rng(10132026)

truth = 0.68
nreal = 1000

single_test = rng.binomial(1,truth,size=nreal)

print('Single realization outcomes')
print('PASS fraction = {:.3f}'.format(np.mean(single_test)))
print('FAIL fraction = {:.3f}'.format(1-np.mean(single_test)))

plt.hist(
    [np.mean(single_test[i:i+1]) for i in range(nreal)],
    bins=[-0.5,0.5,1.5],
    rwidth=0.8
)

plt.xticks([0,1],['Fail','Pass'])
plt.ylabel('Count')
plt.title('A single closure test')
plt.show()

coverage_estimates = []

for seed in range(nreal):

    rng = np.random.default_rng(seed)

    covered = rng.binomial(1,truth,size=100)

    coverage_estimates.append(
        np.mean(covered)
    )

coverage_estimates = np.array(coverage_estimates)

print('\n100-realization coverage study')
print('Mean coverage = {:.3f}'.format(
    np.mean(coverage_estimates)
))
print('Std coverage = {:.3f}'.format(
    np.std(coverage_estimates,ddof=1)
))

plt.hist(coverage_estimates,bins=20)
plt.axvline(
    truth,
    color='r',
    ls='--',
    label='True coverage'
)
plt.xlabel('Measured coverage')
plt.ylabel('Count')
plt.legend()
plt.title('Coverage estimated from many realizations')
plt.show()

**REFEREE REPORT:**

##### 3-sigma cut

The question being answered is if there are two populations present and the sigma clipping treats the data as one population. Removing data based on a single-population estimate can bias the inferred mxture fractions and parameters. In the histogram we can see that members of the high mass tail are permenetly removed, this eliminates potential members of the second population unintentionally basing the mixture model. The proper method would be to include these points in the mixture model.

##### Ignoring Mass error

The dataset provides measured errors for each mas and the AI chose to ignore them in the mixture model. Formally this would be an incorrect method because it treats the masses as exactly known, which they are not. On this specific dataset including the masses has a week effect only changing the shape of the low mass population slightly. $\sigma1$ decreased by 15%

##### False Validation

The AI uses one synthetic dataset to validate the pipline after it finds the true value. This is improper because a $1\sigma$ error bar only garuntees that the true value will be recovered 68% of the time. The proper treatment is to simulate many different datasets and check to see if each parameter is converged on 68% of the time. 

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

I used a custom Microsoft Copilot agent that has been trained to be a ASTR457 study coach. I used the agent to help me understant some of the concepts I was struggling with and the to adopt code from previous labs into this one so I didn't have to retype analysis I already did. I also used the AI to review all of my written work before submitting to check for gramatical and logical errors. To keep the coding style consistent and readable I give the AI old copies of my work to base its style on. As I continued chatting the code became more and more illegible until I had to reset it and change methods for the referee report.

I always have the AI explain its code line by line so that I can understand the process. In doing this I usually catch a few logical errors or bugs that snuck into the ai. If I did not independently verify all of the work in this notebook it probably would have ended up a diconected mess. One such error was the AI swapped between using the variable name "host_logmass" and "host_mass" to reprisent the same value. If I hadn't corrected this the entire notebook would not run in one go because there would be a dimension mismatch after cleaning out the contaminants.

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.